# Semantic Alignment of EEG and Text: A Contrastive Learning Framework for Decoding Imagined Speech

This notebook implements the contrastive EEG–language pipeline used for the CHISCo imagined-speech experiments. It trains one within-subject model per sentence-grouped cross-validation fold, selects the model state with the highest validation Top-5 accuracy, and evaluates only on the corresponding held-out test fold.

The notebook exports the final numerical results and paper-ready figures as **PDF files only**. Intermediate EEG/text caches are kept outside the result directory; trained models, training histories, and split files are not persisted.


## Environment and configuration

The cells below define deterministic execution, portable data/output paths, and the experiment hyperparameters. The default paths still work on Kaggle, while `CHISCO_DATA_ROOT`, `CHISCO_OUTPUT_ROOT`, and `CHISCO_CACHE_ROOT` can be set for local runs without changing the notebook. Required packages are TensorFlow, SentenceTransformers, pandas, scikit-learn, SciPy, Matplotlib, openpyxl, tqdm, and adjustText.


In [ ]:
!pip install -q adjustText
!pip install -q deep-translator

In [ ]:
import os
os.environ["PYTHONHASHSEED"] = "1337"
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"

import gc, hashlib, json, math, pickle, random, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from adjustText import adjust_text
from IPython.display import display
from matplotlib.ticker import PercentFormatter
from scipy.spatial import ConvexHull
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from tensorflow.keras import Model, layers
from tqdm.auto import tqdm

GLOBAL_SEED = 1337

def derive_seed(*parts, base_seed=GLOBAL_SEED):
    payload = "|".join(map(str, (base_seed, *parts))).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:4], "little") & 0x7FFFFFFF

def seed_everything(seed=GLOBAL_SEED):
    random.seed(int(seed)); np.random.seed(int(seed)); tf.keras.utils.set_random_seed(int(seed))

def make_stateless_seed(*parts):
    return np.asarray([derive_seed("stateless_a", *parts), derive_seed("stateless_b", *parts)], dtype=np.int32)

def save_pdf(fig, directory, name):
    path = Path(directory) / f"{name}.pdf"
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    plt.show(); plt.close(fig)
    return path

def mean_std(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if not len(values): return np.nan, np.nan
    return float(values.mean()), float(values.std(ddof=1)) if len(values) > 1 else 0.0

def annotate_bars(ax, bars, values, fmt="{:.3f}"):
    for bar, value in zip(bars, values):
        ax.annotate(fmt.format(value), (bar.get_x() + bar.get_width()/2, bar.get_height()), xytext=(0, 4), textcoords="offset points", ha="center", va="bottom", fontsize=9)

def annotate_points(ax, x, y, fmt="{:.1%}", dx=0, dy=6):
    for xi, yi in zip(x, y):
        ax.annotate(fmt.format(yi), (xi, yi), xytext=(dx, dy), textcoords="offset points", ha="center", fontsize=8.5)

seed_everything()
try: tf.config.experimental.enable_op_determinism()
except Exception: pass
for gpu in tf.config.list_physical_devices("GPU"):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except Exception: pass
tf.config.optimizer.set_jit(False)
print("TensorFlow:", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"), "| Seed:", GLOBAL_SEED)

In [ ]:
SUBJECT_ID = "sub-04"
DEFAULT_DATA_ROOTS = {
    "sub-01": Path("/kaggle/input/datasets/shahryarnamdari/chisco-is-sub01/chisco_sub01_ready"),
    "sub-02": Path("/kaggle/input/datasets/shahryarnamdari/chisco-is-sub02/Chisco-IS-sub02"),
    "sub-03": Path("/kaggle/input/datasets/shahryarnamdari/chisco-is-sub03/Chisco-IS-sub03"),
    "sub-04": Path("/kaggle/input/datasets/shahryarnamdari/chisco-is-sub04/Chisco-IS-sub04"),
    "sub-05": Path("/kaggle/input/datasets/shahryarnamdari/chisco-is-sub05/Chisco-IS-sub05"),
}
DATA_ROOT = Path(os.getenv("CHISCO_DATA_ROOT", str(DEFAULT_DATA_ROOTS[SUBJECT_ID])))
DEFAULT_OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "outputs"
DEFAULT_CACHE_ROOT = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path.cwd() / ".cache"
OUTPUT_ROOT = Path(os.getenv("CHISCO_OUTPUT_ROOT", str(DEFAULT_OUTPUT_ROOT)))
CACHE_ROOT = Path(os.getenv("CHISCO_CACHE_ROOT", str(DEFAULT_CACHE_ROOT)))

RESULTS_DIR = OUTPUT_ROOT / "semantic_alignment" / SUBJECT_ID / "results"
FIGURE_DIR = OUTPUT_ROOT / "semantic_alignment" / SUBJECT_ID / "figures"
CACHE_DIR = CACHE_ROOT / "semantic_alignment" / SUBJECT_ID
for directory in (RESULTS_DIR, FIGURE_DIR, CACHE_DIR): directory.mkdir(parents=True, exist_ok=True)

VAL_RATIO_OF_TRAIN, N_FOLDS = 0.15, 5
NUM_CLASSES, TARGET_C, TARGET_T = 39, 122, 1651
DROP_LAST_N_CHANNELS, PER_TRIAL_ZSCORE = 3, True
PKL_FILE_KEYWORD = "imagine"
TEXT_MODEL_NAME, TEXT_EMBEDDING_DEVICE = "BAAI/bge-small-zh-v1.5", "cpu"
EEG_CACHE_DTYPE = np.float16

TRAIN_BATCH_SIZE, EVAL_BATCH_SIZE = 64, 128
MAX_EPOCHS, LEARNING_RATE = 100, 3e-4
EARLY_STOPPING_PATIENCE, MIN_DELTA = 16, 1e-3
REDUCE_LR_PATIENCE, REDUCE_LR_FACTOR, MIN_LR = 5, 0.7, 1e-6
INITIAL_TAU, TAU_MIN, TAU_MAX = 0.07, 0.01, 1.0
LAMBDA_INST, LAMBDA_CLS = 1.0, 0.5
LABEL_SMOOTHING, SEMANTIC_NEGATIVE_FLOOR = 0.1, 0.05
AUG_NOISE_STD, AUG_CHANNEL_DROPOUT_RATE = 0.05, 0.10
AUG_CHANNEL_DROPOUT_PROB, AUG_MAX_TEMPORAL_SHIFT = 0.10, 50
TOPK_LIST, N_RANDOM_BASELINE, N_2V2_PAIRS = [1, 2, 3, 5, 10], 10_000, 20_000

print("Subject:", SUBJECT_ID)
print("Data root:", DATA_ROOT)
print("Results:", RESULTS_DIR)
print("Figures:", FIGURE_DIR)

## Dataset and preprocessing

This section reconstructs the CHISCo class taxonomy, maps each imagined-speech trial to its semantic category, and applies the same trial-level preprocessing used in the experiment: removal of the last three non-EEG channels, per-trial z-scoring, and center cropping/padding to 122 × 1651 samples. Preprocessed EEG is cached as float16 for efficient repeated access, while TensorFlow batches are converted back to float32 during training.


In [ ]:
CLASS_ZH_TO_EN = {
    '预订和旅行安排': 'Travel Arrangements', '住房和设施': 'Housing and Facilities', '自然和天气': 'Nature and Weather',
    '个人行为和日常活动': 'Personal Behavior and Daily Activities', '金融和付款': 'Finance', '饮食和用餐': 'Food and Dining',
    '旅行和行李管理': 'Travel Affairs', '交通和出行': 'Transportation and Commuting', '旅游和度假': 'Vacation',
    '设备故障和环境问题': 'Equipment Malfunction or Environmental Issues', '价格和费用': 'Prices and Costs',
    '时间和日程安排': 'Time and Scheduling', '衣物和服饰': 'Clothing', '语言和学习': 'Learning',
    '欢迎和感谢': 'Welcoming and Thanking', '道歉和请求原谅': 'Apologies', '个人信息': 'Personal Information',
    '询问和个人事务': 'Inquiries and Personal Matters', '饮食习惯': 'Eating Habits', '产品和质量保证': 'Products and Quality Assurance',
    '健康和安全建议': 'Health and Safety', '家庭关系和家庭事件': 'Family Relationships and Events', '提供和请求帮助': 'Providing and Requesting Assistance',
    '理发和美容护理': 'Hairdressing and Beauty Care', '节日和庆祝活动': 'Festivals and Celebrations',
    '互联网和信息技术': 'Internet and Information Technology', '健康和身体不适': 'Physical Discomfort',
    '情感和人际关系': 'Emotions and Interpersonal Relationships', '社交和聚会活动': 'Gathering Activities',
    '人际交往和情感表达': 'Social Interactions', '问候和情感状态': 'Greetings', '工作和职场交流': 'Work',
    '表演艺术': 'Performing Arts', '教育和学习': 'Education', '娱乐和媒体消费': 'Entertainment and Media Consumption',
    '求职和职业发展': 'Job Hunting and Career Development', '健身': 'Fitness', '兴趣爱好': 'Hobbies', '体育和运动': 'Sports'
}
CLASS_PHRASES_ZH = list(CLASS_ZH_TO_EN)
CLASS_PHRASES_EN = [CLASS_ZH_TO_EN[x] for x in CLASS_PHRASES_ZH]
CLASS_ZH_TO_ID = {name: i for i, name in enumerate(CLASS_PHRASES_ZH)}
CLASS_ID_TO_ZH = {i: name for name, i in CLASS_ZH_TO_ID.items()}
CLASS_ID_TO_EN = {i: CLASS_ZH_TO_EN[name] for name, i in CLASS_ZH_TO_ID.items()}
assert len(CLASS_PHRASES_ZH) == NUM_CLASSES and CLASS_ID_TO_EN[36] == "Fitness" and CLASS_ID_TO_EN[38] == "Sports"

def find_dataset_root(root):
    root = Path(root)
    if (root / "textdataset").exists() and (root / "derivatives" / "preprocessed_pkl").exists(): return root
    matches = [p.parent for p in root.rglob("textdataset") if (p.parent / "derivatives" / "preprocessed_pkl").exists()]
    if not matches: raise FileNotFoundError(f"Could not find CHISCo data under {root}")
    return matches[0]

def _find_column(columns, tokens, default=None):
    return next((c for c in columns if any(t.lower() in c.lower() for t in tokens)), default)

def load_text_metadata(text_dir):
    frames = [pd.read_excel(path) for path in sorted(Path(text_dir).glob("*.xlsx"))]
    frames = [frame for frame in frames if frame.shape[1] >= 2]
    if not frames: raise FileNotFoundError(f"No valid textdataset Excel files in {text_dir}")
    frame = pd.concat(frames, ignore_index=True); columns = [str(c) for c in frame.columns]
    sentence_col = _find_column(columns, ["句子", "sentence", "text"], columns[0])
    label_col = _find_column(columns, ["标签", "label", "class"], columns[1])
    english_col = _find_column([c for c in columns if c != sentence_col], ["sentence_en", "english sentence", "translation", "english", "英文翻译", "英语翻译"])
    sentences = frame[sentence_col].astype(str).str.strip(); labels = frame[label_col].astype(str).str.strip()
    text_to_class = {s: l for s, l in zip(sentences, labels) if l in CLASS_ZH_TO_ID}
    text_to_english = {}
    if english_col is not None:
        for zh, en in zip(sentences, frame[english_col]):
            en = str(en).strip()
            if zh and en and en.lower() != "nan": text_to_english[zh] = en
    if not text_to_class: raise RuntimeError("No sentences mapped to the 39 CHISCo classes.")
    return text_to_class, text_to_english

DATA_ROOT = find_dataset_root(DATA_ROOT)
TEXTDATASET_DIR = DATA_ROOT / "textdataset"
PKL_DIR = DATA_ROOT / "derivatives" / "preprocessed_pkl"
EEG_PATHS = sorted(p for p in PKL_DIR.rglob("*.pkl") if PKL_FILE_KEYWORD.lower() in p.name.lower())
if not EEG_PATHS: raise RuntimeError(f"No imagined-speech pickle files found under {PKL_DIR}")
SENTENCE_TO_CLASS_ZH, TEXT_TRANSLATIONS = load_text_metadata(TEXTDATASET_DIR)
print("EEG files:", len(EEG_PATHS), "| mapped sentences:", len(SENTENCE_TO_CLASS_ZH), "| English translations:", len(TEXT_TRANSLATIONS))

In [ ]:
def load_pickle_list(path):
    with open(path, "rb") as f: data = pickle.load(f)
    if not isinstance(data, list): raise TypeError(f"Expected list in {path}, got {type(data)}")
    return data

def preprocess_eeg(sample_input):
    x = np.asarray(sample_input)
    if x.ndim == 3 and x.shape[0] == 1: x = np.squeeze(x, axis=0)
    if x.ndim == 3 and x.shape[-1] == 1: x = np.squeeze(x, axis=-1)
    if x.ndim != 2: raise ValueError(f"Unexpected EEG shape: {x.shape}")
    if x.shape[0] > 300 and x.shape[1] <= 512: x = x.T
    x = x.astype(np.float32, copy=False)
    if x.shape[0] >= TARGET_C + DROP_LAST_N_CHANNELS: x = x[:-DROP_LAST_N_CHANNELS]
    else:
        x = x[:min(TARGET_C, x.shape[0])]
        if x.shape[0] < TARGET_C: x = np.pad(x, ((0, TARGET_C - x.shape[0]), (0, 0)))
    if PER_TRIAL_ZSCORE: x = (x - x.mean(keepdims=True)) / (x.std(keepdims=True) + 1e-6)
    if x.shape[1] > TARGET_T:
        start = (x.shape[1] - TARGET_T) // 2; x = x[:, start:start + TARGET_T]
    elif x.shape[1] < TARGET_T:
        pad = TARGET_T - x.shape[1]; x = np.pad(x, ((0, 0), (pad // 2, pad - pad // 2)))
    return np.nan_to_num(x).astype(np.float32)

def collect_trial_index(paths):
    rows, skipped_text, skipped_sample = [], 0, 0
    for path in tqdm(paths, desc="Indexing EEG trials"):
        samples = load_pickle_list(path)
        for sample_idx, sample in enumerate(samples):
            sentence = str(sample.get("text", "")).strip(); label_zh = SENTENCE_TO_CLASS_ZH.get(sentence)
            if label_zh not in CLASS_ZH_TO_ID: skipped_text += 1; continue
            if "input_features" not in sample: skipped_sample += 1; continue
            rows.append({"trial_uid": len(rows), "pkl_path": str(path), "pkl_file": path.name, "sample_idx": sample_idx,
                         "sentence_zh": sentence, "label_zh": label_zh, "label_en": CLASS_ZH_TO_EN[label_zh], "label_id": CLASS_ZH_TO_ID[label_zh]})
    index = pd.DataFrame(rows)
    if index.empty: raise RuntimeError("No valid EEG trials were collected.")
    print(f"Trials: {len(index)} | skipped text: {skipped_text} | skipped samples: {skipped_sample} | classes: {index.label_id.nunique()}")
    return index

def load_eeg_cache(index):
    shape = (len(index), TARGET_C, TARGET_T)
    tag = "trialzscore" if PER_TRIAL_ZSCORE else "raw"
    path = CACHE_DIR / f"eeg_{tag}_n{shape[0]}_{TARGET_C}x{TARGET_T}_{np.dtype(EEG_CACHE_DTYPE).name}.dat"
    expected_bytes = int(np.prod(shape) * np.dtype(EEG_CACHE_DTYPE).itemsize)
    if not path.exists() or path.stat().st_size != expected_bytes:
        if path.exists(): path.unlink()
        mm = np.memmap(path, dtype=EEG_CACHE_DTYPE, mode="w+", shape=shape)
        for pkl_path, group in tqdm(index.groupby("pkl_path", sort=False), desc="Preprocessing EEG"):
            samples = load_pickle_list(Path(pkl_path))
            for row in group.itertuples(): mm[row.trial_uid] = preprocess_eeg(samples[row.sample_idx]["input_features"]).astype(EEG_CACHE_DTYPE)
            mm.flush(); del samples; gc.collect()
        del mm; gc.collect()
    return np.memmap(path, dtype=EEG_CACHE_DTYPE, mode="r", shape=shape)

trial_index = collect_trial_index(EEG_PATHS)
y_class = trial_index["label_id"].to_numpy(np.int64)
X_EEG_MEMMAP = load_eeg_cache(trial_index)
X_EEG = np.asarray(X_EEG_MEMMAP, dtype=np.float16)
print("EEG tensor:", X_EEG.shape, X_EEG.dtype, "| class count range:", np.bincount(y_class, minlength=NUM_CLASSES).min(), "-", np.bincount(y_class, minlength=NUM_CLASSES).max())
display(trial_index.head(3))

## Frozen text representations

The frozen `BAAI/bge-small-zh-v1.5` encoder defines the 512-dimensional semantic target space. Original Chinese stimulus sentences are embedded as instance-level targets, and the 39 Chinese class names are embedded as fixed category prototypes; both are L2-normalized and cached locally.


In [ ]:
from sentence_transformers import SentenceTransformer

def l2_normalize_np(x, axis=1, eps=1e-12):
    return x / np.maximum(np.linalg.norm(x, axis=axis, keepdims=True), eps)

def load_text_embeddings():
    path = CACHE_DIR / "bge_sentence_and_class_embeddings.npz"
    sentences = sorted(SENTENCE_TO_CLASS_ZH)
    if path.exists():
        data = np.load(path, allow_pickle=True)
        if str(data["model_name"]) == TEXT_MODEL_NAME and data["sentences"].tolist() == sentences:
            return sentences, data["sentence_emb"].astype(np.float32), data["class_emb"].astype(np.float32)
    model = SentenceTransformer(TEXT_MODEL_NAME, device=TEXT_EMBEDDING_DEVICE)
    sentence_emb = model.encode(sentences, batch_size=256, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True).astype(np.float32)
    class_emb = model.encode(CLASS_PHRASES_ZH, batch_size=256, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True).astype(np.float32)
    sentence_emb, class_emb = l2_normalize_np(sentence_emb), l2_normalize_np(class_emb)
    np.savez(path, model_name=np.array(TEXT_MODEL_NAME), sentences=np.array(sentences, dtype=object), sentence_emb=sentence_emb, class_emb=class_emb)
    return sentences, sentence_emb, class_emb

all_sentences, sentence_emb, class_emb = load_text_embeddings()
SENTENCE_TO_EMBEDDING = dict(zip(all_sentences, sentence_emb))
y_text = np.vstack([SENTENCE_TO_EMBEDDING[s] for s in trial_index["sentence_zh"]]).astype(np.float32)
EMB_DIM = class_emb.shape[1]
print("Sentence embeddings:", sentence_emb.shape, "| class prototypes:", class_emb.shape)

## Sentence-grouped cross-validation

All trials belonging to the same sentence remain in the same split. The outer protocol uses stratified five-fold cross-validation over unique sentence groups; within each training pool, 15% of sentence groups are reserved for validation with the same fixed seed. The validation and leakage checks below reproduce the original split logic but do not write split files to the result directory.


In [ ]:
def make_sentence_groups(index):
    if (index.groupby("sentence_zh")["label_id"].nunique() > 1).any(): raise ValueError("A sentence maps to multiple classes.")
    groups = index.groupby("sentence_zh", sort=True).agg(label_id=("label_id", "first"), label_en=("label_en", "first"), n_trials=("trial_uid", "size")).reset_index()
    groups["group_id"] = np.arange(len(groups), dtype=np.int64)
    mapping = dict(zip(groups["sentence_zh"], groups["group_id"]))
    return groups, index["sentence_zh"].map(mapping).to_numpy(np.int64)

def build_cv_splits(index):
    groups, trial_group_ids = make_sentence_groups(index)
    group_ids = groups["group_id"].to_numpy(np.int64); group_labels = groups["label_id"].to_numpy(np.int64)
    if np.bincount(group_labels, minlength=NUM_CLASSES).min() < N_FOLDS: raise ValueError("Too few sentence groups per class for five-fold CV.")
    outer = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=GLOBAL_SEED)
    splits = []
    for fold_id, (trainval_pos, test_pos) in enumerate(outer.split(group_ids, group_labels)):
        trainval_ids, test_ids = group_ids[trainval_pos], group_ids[test_pos]
        trainval_labels = group_labels[trainval_pos]
        inner = StratifiedShuffleSplit(n_splits=1, test_size=VAL_RATIO_OF_TRAIN, random_state=GLOBAL_SEED)
        train_rel, val_rel = next(inner.split(trainval_ids, trainval_labels))
        train_ids, val_ids = trainval_ids[train_rel], trainval_ids[val_rel]
        idx = lambda selected: np.flatnonzero(np.isin(trial_group_ids, selected)).astype(np.int64)
        splits.append({"fold_id": fold_id, "train_idx": idx(train_ids), "val_idx": idx(val_ids), "test_idx": idx(test_ids),
                       "train_group_ids": train_ids, "val_group_ids": val_ids, "test_group_ids": test_ids})
    return splits, groups

def validate_cv_splits(index, splits):
    all_trials = np.arange(len(index), dtype=np.int64); pooled_test = []
    for split in splits:
        train, val, test = split["train_idx"], split["val_idx"], split["test_idx"]
        if any(np.intersect1d(a, b).size for a, b in ((train, val), (train, test), (val, test))): raise AssertionError(f"Fold {split['fold_id']}: trial overlap")
        if not np.array_equal(np.sort(np.concatenate([train, val, test])), all_trials): raise AssertionError(f"Fold {split['fold_id']}: incomplete trial coverage")
        sentence_sets = [set(index.iloc[idx]["sentence_zh"]) for idx in (train, val, test)]
        if sentence_sets[0] & sentence_sets[1] or sentence_sets[0] & sentence_sets[2] or sentence_sets[1] & sentence_sets[2]: raise AssertionError(f"Fold {split['fold_id']}: sentence leakage")
        pooled_test.append(test)
    if not np.array_equal(np.sort(np.concatenate(pooled_test)), all_trials): raise AssertionError("Each trial must occur in exactly one outer test fold.")

CV_SPLITS, sentence_groups = build_cv_splits(trial_index)
validate_cv_splits(trial_index, CV_SPLITS)
summary = pd.DataFrame([{"fold": s["fold_id"], "train_trials": len(s["train_idx"]), "validation_trials": len(s["val_idx"]), "test_trials": len(s["test_idx"]),
                         "train_sentences": len(s["train_group_ids"]), "validation_sentences": len(s["val_group_ids"]), "test_sentences": len(s["test_group_ids"])} for s in CV_SPLITS])
display(summary)

## EEG encoder and training objective

The EEG encoder follows the paper architecture: an EEGNet-style convolutional front end, sinusoidal positional encoding, four Transformer blocks, attention pooling, and a projection head into the frozen text space. Training combines soft-weighted InfoNCE sentence alignment with prototype-based weighted cross-entropy, using the same class weighting, label smoothing, learned temperature, and EEG augmentations as the original experiment.


In [ ]:
def eeg_feature_extractor(inputs, dropout_rate=0.1):
    x = layers.Conv2D(8, (1, 125), padding="same", use_bias=False, name="layer1_conv")(inputs)
    x = layers.LayerNormalization(axis=-1, epsilon=1e-6, name="layer1_ln")(x)
    x = layers.DepthwiseConv2D((TARGET_C, 1), use_bias=False, depth_multiplier=8, name="layer1_depthwise")(x)
    x = layers.LayerNormalization(axis=-1, epsilon=1e-6, name="layer1_depthwise_ln")(x)
    x = layers.Activation("elu", name="layer1_elu")(x); x = layers.AveragePooling2D((1, 2), name="layer1_pool")(x); x = layers.Dropout(dropout_rate)(x)
    x = layers.SeparableConv2D(64, (1, 25), use_bias=False, padding="same", name="layer2_sep")(x)
    x = layers.LayerNormalization(axis=-1, epsilon=1e-6, name="layer2_ln")(x)
    x = layers.Activation("elu", name="layer2_elu")(x); x = layers.AveragePooling2D((1, 5), name="layer2_pool")(x); x = layers.Dropout(dropout_rate)(x)
    return layers.Lambda(lambda t: tf.squeeze(t, axis=1), name="squeeze_channels")(x)

class PositionalEncoding(layers.Layer):
    def call(self, x):
        seq_len, d_model = tf.shape(x)[1], tf.shape(x)[2]
        position = tf.cast(tf.range(seq_len)[:, None], tf.float32)
        div_term = tf.exp(tf.range(0, d_model, 2, dtype=tf.float32) * -(math.log(10000.0) / tf.cast(d_model, tf.float32)))
        pe = tf.concat([tf.sin(position * div_term), tf.cos(position * div_term)], axis=-1)[:, :d_model]
        return x + pe[None, ...]

def transformer_block(x, num_heads=8, key_dim=64, ff_dim=256, dropout=0.1):
    attention = layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim, dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + attention)
    ffn = layers.Dense(ff_dim, activation="gelu")(x); ffn = layers.Dropout(dropout)(ffn); ffn = layers.Dense(key_dim)(ffn)
    return layers.LayerNormalization(epsilon=1e-6)(x + ffn)

class AttentionPooling1D(layers.Layer):
    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1), initializer="glorot_uniform", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(1,), initializer="zeros", trainable=True)
        super().build(input_shape)
    def call(self, inputs):
        scores = tf.squeeze(tf.tanh(tf.matmul(inputs, self.W) + self.b), axis=-1)
        weights = tf.nn.softmax(scores, axis=1)
        return tf.reduce_sum(inputs * weights[..., None], axis=1)

def build_eeg_encoder(input_shape=(TARGET_C, TARGET_T, 1), output_dim=EMB_DIM, cnn_dropout=0.5, trans_dropout=0.1):
    inputs = layers.Input(shape=input_shape, name="eeg_input")
    x = eeg_feature_extractor(inputs, cnn_dropout); x = PositionalEncoding()(x); x = layers.Dropout(trans_dropout)(x)
    for _ in range(4): x = transformer_block(x, num_heads=8, key_dim=64, ff_dim=256, dropout=trans_dropout)
    x = AttentionPooling1D()(x); x = layers.Dense(256, activation="gelu")(x); x = layers.Dropout(trans_dropout)(x)
    x = layers.Dense(output_dim, name="projection_head")(x)
    x = layers.Lambda(lambda t: tf.math.l2_normalize(t, axis=-1), name="eeg_emb_norm")(x)
    return Model(inputs, x, name="CHISCo_Contrastive_Encoder")

encoder_preview = build_eeg_encoder(); encoder_preview.summary(); del encoder_preview; gc.collect()

In [ ]:
def class_weights(y_train):
    counts = np.bincount(y_train, minlength=NUM_CLASSES).astype(np.float32)
    freq = (counts + 1e-6) / (counts.sum() + 1e-6)
    weights = (1 / np.sqrt(freq)); weights /= weights.mean()
    return tf.constant(weights, dtype=tf.float32), counts.astype(int)

def iter_batches(indices, batch_size, training=False, drop_remainder=False, shuffle_seed=None):
    order = np.asarray(indices, dtype=np.int64)
    if training:
        if shuffle_seed is None: raise ValueError("shuffle_seed is required for training batches")
        order = np.random.default_rng(int(shuffle_seed)).permutation(order)
    else: order = order.copy()
    if drop_remainder: order = order[:(len(order) // batch_size) * batch_size]
    for start in range(0, len(order), batch_size):
        idx = order[start:start + batch_size]
        yield X_EEG[idx].astype(np.float32, copy=False)[..., None], y_text[idx].astype(np.float32, copy=False), y_class[idx].astype(np.int64, copy=False)

@tf.function
def l2_normalize_tf(x): return tf.math.l2_normalize(tf.cast(x, tf.float32), axis=-1)

@tf.function
def gather_weights(labels, weights):
    w = tf.gather(weights, labels)
    return w / (tf.reduce_mean(w) + 1e-8)

@tf.function
def augment_eeg_tf(x, stateless_seed):
    x = tf.cast(x, tf.float32); seeds = tf.random.experimental.stateless_split(tf.cast(stateless_seed, tf.int32), 4)
    x += tf.random.stateless_normal(tf.shape(x), seed=seeds[0], stddev=AUG_NOISE_STD, dtype=tf.float32)
    do_drop = tf.random.stateless_uniform([], seed=seeds[1]) < AUG_CHANNEL_DROPOUT_PROB
    def channel_dropout():
        shape = tf.stack([tf.shape(x)[0], TARGET_C, 1, 1])
        keep = tf.cast(tf.random.stateless_uniform(shape, seed=seeds[2]) > AUG_CHANNEL_DROPOUT_RATE, tf.float32)
        return x * keep
    x = tf.cond(do_drop, channel_dropout, lambda: x)
    shift = tf.random.stateless_uniform([], seed=seeds[3], minval=-AUG_MAX_TEMPORAL_SHIFT, maxval=AUG_MAX_TEMPORAL_SHIFT + 1, dtype=tf.int32)
    return tf.roll(x, shift=shift, axis=2)

@tf.function
def cosine_logits(eeg_z, prototypes, tau): return tf.matmul(l2_normalize_tf(eeg_z), l2_normalize_tf(prototypes), transpose_b=True) / tau

@tf.function
def info_nce_loss(eeg_z, text_z, labels, weights, tau):
    z, t = l2_normalize_tf(eeg_z), l2_normalize_tf(text_z); batch_size = tf.shape(z)[0]
    logits = tf.matmul(z, t, transpose_b=True) / tau
    semantic_weights = tf.maximum(SEMANTIC_NEGATIVE_FLOOR, 1.0 - tf.matmul(t, t, transpose_b=True))
    eye = tf.eye(batch_size); weight_matrix = eye + (1.0 - eye) * semantic_weights
    logits_max = tf.stop_gradient(tf.reduce_max(logits, axis=1, keepdims=True)); shifted = logits - logits_max
    log_denom = tf.math.log(tf.reduce_sum(weight_matrix * tf.exp(shifted), axis=1) + 1e-9) + tf.squeeze(logits_max, axis=1)
    return tf.reduce_mean(-(tf.linalg.diag_part(logits) - log_denom) * gather_weights(labels, weights))

@tf.function
def class_loss(logits, labels, weights):
    targets = tf.one_hot(labels, depth=NUM_CLASSES)
    ce = tf.keras.losses.categorical_crossentropy(targets, logits, from_logits=True, label_smoothing=LABEL_SMOOTHING)
    return tf.reduce_mean(ce * gather_weights(labels, weights))

CLASS_PROTOTYPES = tf.constant(class_emb.astype(np.float32), dtype=tf.float32)

## Training and held-out encoding

Each fold is trained from scratch with a deterministic fold seed. Early stopping and learning-rate reduction are driven by validation loss, while model selection for the reported contrastive results uses the highest validation Top-5 accuracy, exactly as in the experiment. The selected weights are kept only in memory long enough to encode that fold's validation and test trials, then the model is discarded; no model file is written.


In [ ]:
def encode_indices(encoder, indices, batch_size=EVAL_BATCH_SIZE):
    z, labels = [], []
    iterator = iter_batches(indices, batch_size, training=False)
    for x, _, y in tqdm(iterator, total=int(np.ceil(len(indices) / batch_size)), leave=False):
        z.append(encoder(x, training=False).numpy().astype(np.float32)); labels.append(y)
    return np.concatenate(z), np.concatenate(labels)

def eeg_class_cosine(eeg_z):
    return np.clip(l2_normalize_np(np.asarray(eeg_z, dtype=np.float32)) @ l2_normalize_np(class_emb).T, -1.0, 1.0).astype(np.float32)

def train_fold(split):
    fold_id = int(split["fold_id"]); train_idx, val_idx, test_idx = split["train_idx"], split["val_idx"], split["test_idx"]
    fold_seed = derive_seed("model_and_tf", fold_id); weights, counts = class_weights(y_class[train_idx])
    print(f"\nFold {fold_id} | train {len(train_idx)} | validation {len(val_idx)} | test {len(test_idx)} | class count {counts.min()}-{counts.max()}")
    tf.keras.backend.clear_session(); gc.collect(); seed_everything(fold_seed)
    encoder = build_eeg_encoder(cnn_dropout=0.5, trans_dropout=0.1)
    optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    log_temp = tf.Variable(tf.math.log(tf.constant(INITIAL_TAU, tf.float32)), trainable=True, name="log_temperature")
    train_metrics = [tf.keras.metrics.Mean() for _ in range(3)] + [tf.keras.metrics.SparseCategoricalAccuracy(), tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5)]
    val_metrics = [tf.keras.metrics.Mean() for _ in range(3)] + [tf.keras.metrics.SparseCategoricalAccuracy(), tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5)]

    @tf.function(reduce_retracing=True)
    def train_step(x, text_z, labels, aug_seed):
        tau = tf.clip_by_value(tf.exp(log_temp), TAU_MIN, TAU_MAX); x = augment_eeg_tf(x, aug_seed)
        with tf.GradientTape() as tape:
            eeg_z = encoder(x, training=True); logits = cosine_logits(eeg_z, CLASS_PROTOTYPES, tau)
            inst = info_nce_loss(eeg_z, text_z, labels, weights, tau); cls = class_loss(logits, labels, weights); total = LAMBDA_INST * inst + LAMBDA_CLS * cls
        variables = encoder.trainable_variables + [log_temp]
        gradients = tape.gradient(total, variables)
        optimizer.apply_gradients((g, v) for g, v in zip(gradients, variables) if g is not None)
        for metric, value in zip(train_metrics[:3], (total, inst, cls)): metric.update_state(value)
        train_metrics[3].update_state(labels, logits); train_metrics[4].update_state(labels, logits)

    @tf.function(reduce_retracing=True)
    def val_step(x, text_z, labels):
        tau = tf.clip_by_value(tf.exp(log_temp), TAU_MIN, TAU_MAX); eeg_z = encoder(x, training=False); logits = cosine_logits(eeg_z, CLASS_PROTOTYPES, tau)
        inst = info_nce_loss(eeg_z, text_z, labels, weights, tau); cls = class_loss(logits, labels, weights); total = LAMBDA_INST * inst + LAMBDA_CLS * cls
        for metric, value in zip(val_metrics[:3], (total, inst, cls)): metric.update_state(value)
        val_metrics[3].update_state(labels, logits); val_metrics[4].update_state(labels, logits)

    best_loss, best_lr_loss, best_top5 = np.inf, np.inf, -np.inf
    bad_es = bad_lr = 0; best_top5_weights = None
    train_steps = len(train_idx) // TRAIN_BATCH_SIZE; val_steps = int(np.ceil(len(val_idx) / TRAIN_BATCH_SIZE))
    for epoch in range(1, MAX_EPOCHS + 1):
        t0 = time.time()
        for metric in (*train_metrics, *val_metrics): metric.reset_state()
        shuffle_seed = derive_seed("batch_order", fold_id, epoch)
        train_iter = iter_batches(train_idx, TRAIN_BATCH_SIZE, training=True, drop_remainder=True, shuffle_seed=shuffle_seed)
        for batch_id, (x, text_z, labels) in enumerate(tqdm(train_iter, total=train_steps, desc=f"fold {fold_id} epoch {epoch:03d} train", leave=False)):
            train_step(x, text_z, labels, make_stateless_seed("augmentation", fold_id, epoch, batch_id))
        val_iter = iter_batches(val_idx, TRAIN_BATCH_SIZE)
        for x, text_z, labels in tqdm(val_iter, total=val_steps, desc=f"fold {fold_id} epoch {epoch:03d} validation", leave=False): val_step(x, text_z, labels)
        val_loss = float(val_metrics[0].result()); val_top5 = float(val_metrics[4].result()); notes = []
        if val_loss < best_loss - MIN_DELTA:
            best_loss = val_loss; best_lr_loss = val_loss; bad_es = bad_lr = 0
        else:
            bad_es += 1
            if val_loss < best_lr_loss - MIN_DELTA: best_lr_loss = val_loss; bad_lr = 0
            else: bad_lr += 1
        if bad_lr >= REDUCE_LR_PATIENCE:
            old_lr = float(optimizer.learning_rate.numpy()); new_lr = max(old_lr * REDUCE_LR_FACTOR, MIN_LR)
            optimizer.learning_rate.assign(new_lr); bad_lr = 0; notes.append(f"lr {old_lr:.2e}->{new_lr:.2e}")
        if val_top5 > best_top5 + MIN_DELTA:
            best_top5 = val_top5; best_top5_weights = [w.copy() for w in encoder.get_weights()]; notes.append("best val Top-5")
        print(f"Epoch {epoch:03d} | loss {float(train_metrics[0].result()):.4f}/{val_loss:.4f} | Top-1 {float(train_metrics[3].result()):.3f}/{float(val_metrics[3].result()):.3f} | Top-5 {float(train_metrics[4].result()):.3f}/{val_top5:.3f} | tau {float(tf.clip_by_value(tf.exp(log_temp), TAU_MIN, TAU_MAX)):.4f} | {time.time()-t0:.1f}s" + (" | " + "; ".join(notes) if notes else ""))
        if bad_es >= EARLY_STOPPING_PATIENCE: print(f"Early stopping at epoch {epoch}"); break

    if best_top5_weights is None: raise RuntimeError(f"Fold {fold_id}: no validation Top-5 model state was selected.")
    encoder.set_weights(best_top5_weights)
    val_z, val_y = encode_indices(encoder, val_idx); test_z, test_y = encode_indices(encoder, test_idx)
    val_cosine, test_cosine = eeg_class_cosine(val_z), eeg_class_cosine(test_z)

    result = {"fold_id": fold_id, "train_idx": np.asarray(train_idx), "val_idx": np.asarray(val_idx), "test_idx": np.asarray(test_idx),
              "val_z": val_z, "val_y": val_y, "val_cosine": val_cosine, "val_ranked": np.argsort(-val_cosine, axis=1),
              "test_z": test_z, "test_y": test_y, "test_cosine": test_cosine, "test_ranked": np.argsort(-test_cosine, axis=1),
              "best_val_top5": best_top5}
    del encoder, best_top5_weights; tf.keras.backend.clear_session(); gc.collect()
    return result

In [ ]:
fold_results = [train_fold(split) for split in CV_SPLITS]
pooled_test = {
    "eeg_z": np.concatenate([item["test_z"] for item in fold_results]),
    "y_true": np.concatenate([item["test_y"] for item in fold_results]),
    "fold_id": np.concatenate([np.full(len(item["test_y"]), item["fold_id"], dtype=np.int64) for item in fold_results]),
}
print("Held-out test embeddings:", pooled_test["eeg_z"].shape)

## Decoding performance

Held-out decoding is evaluated with Top-k accuracy, Macro-F1, and balanced class-level 2v2 accuracy. The prior-aware random baseline samples predictions according to the fold-specific training-class distribution with 10,000 Monte Carlo runs; 2v2 uses 20,000 class-balanced pairs per fold with a theoretical chance level of 0.50. Figures report fold means as numeric labels and represent fold variability only through **std** error bars.


In [ ]:
PAPER_COLORS = {"model": "#3B6EA8", "random": "#A9A9A9"}

def topk_accuracy(y, ranked, k): return float((ranked[:, :k] == np.asarray(y)[:, None]).any(axis=1).mean())

def macro_f1_many(y, predictions):
    y, predictions = np.asarray(y, dtype=np.int64), np.asarray(predictions, dtype=np.int64)
    scores = np.zeros(predictions.shape[0], dtype=float)
    for class_id in range(NUM_CLASSES):
        true = y == class_id; tp = (predictions[:, true] == class_id).sum(axis=1); fp = (predictions[:, ~true] == class_id).sum(axis=1); fn = true.sum() - tp
        denom = 2*tp + fp + fn; scores += np.divide(2*tp, denom, out=np.zeros_like(tp, dtype=float), where=denom > 0)
    return scores / NUM_CLASSES

def prior_aware_random_metrics(y, class_prior, seed):
    rng = np.random.default_rng(seed); y = np.asarray(y, dtype=np.int64); prior = np.asarray(class_prior, dtype=np.float32); prior /= prior.sum()
    max_k = max(TOPK_LIST); bytes_per_run = len(y) * len(prior) * 4; chunk = int(np.clip((80 * 1024**2) / max(bytes_per_run, 1), 20, 500)); log_prior = np.log(prior + 1e-12)
    topk_runs, macro_runs = {k: [] for k in TOPK_LIST}, []
    for start in tqdm(range(0, N_RANDOM_BASELINE, chunk), desc="Prior-aware random baseline", leave=False):
        n = min(chunk, N_RANDOM_BASELINE - start); scores = rng.gumbel(size=(n, len(y), len(prior))).astype(np.float32) + log_prior[None, None, :]
        top = np.argpartition(scores, -max_k, axis=2)[:, :, -max_k:]; order = np.argsort(-np.take_along_axis(scores, top, axis=2), axis=2); ranked = np.take_along_axis(top, order, axis=2)
        for k in TOPK_LIST: topk_runs[k].append((ranked[:, :, :k] == y[None, :, None]).any(axis=2).mean(axis=1))
        macro_runs.append(macro_f1_many(y, ranked[:, :, 0]))
    result = {f"top{k}": mean_std(np.concatenate(topk_runs[k])) for k in TOPK_LIST}; result["macro_f1"] = mean_std(np.concatenate(macro_runs))
    return result

metric_rows = []
for item in tqdm(fold_results, desc="Decoding metrics"):
    y, ranked = item["test_y"], item["test_ranked"]; pred = ranked[:, 0]
    prior = np.bincount(y_class[item["train_idx"]], minlength=NUM_CLASSES).astype(np.float32)
    random_stats = prior_aware_random_metrics(y, prior, GLOBAL_SEED + 1000 * item["fold_id"])
    row = {"fold_id": item["fold_id"], "n_test": len(y), "macro_f1": f1_score(y, pred, labels=np.arange(NUM_CLASSES), average="macro", zero_division=0),
           "random_macro_f1_mean": random_stats["macro_f1"][0], "random_macro_f1_std": random_stats["macro_f1"][1]}
    for k in TOPK_LIST:
        row[f"top{k}"] = topk_accuracy(y, ranked, k); row[f"random_top{k}_mean"], row[f"random_top{k}_std"] = random_stats[f"top{k}"]
    metric_rows.append(row)
fold_metrics = pd.DataFrame(metric_rows)
summary_rows = []
for metric in [f"top{k}" for k in TOPK_LIST] + ["macro_f1"]:
    mean, std = mean_std(fold_metrics[metric]); summary_rows.append({"metric": metric, "mean": mean, "std": std, "n_folds": N_FOLDS})
metric_summary = pd.DataFrame(summary_rows)
fold_metrics.to_csv(RESULTS_DIR / "decoding_metrics_by_fold.csv", index=False); metric_summary.to_csv(RESULTS_DIR / "decoding_metrics_summary.csv", index=False)
display(fold_metrics); display(metric_summary)

x = np.arange(len(TOPK_LIST)); width = 0.36
model_mean = np.array([fold_metrics[f"top{k}"].mean() for k in TOPK_LIST]); model_std = np.array([fold_metrics[f"top{k}"].std(ddof=1) for k in TOPK_LIST])
random_mean = np.array([fold_metrics[f"random_top{k}_mean"].mean() for k in TOPK_LIST]); random_std = np.array([fold_metrics[f"random_top{k}_mean"].std(ddof=1) for k in TOPK_LIST])
fig, ax = plt.subplots(figsize=(9.5, 5.4))
b1 = ax.bar(x-width/2, model_mean, width, yerr=model_std, capsize=4, label="Contrastive", color=PAPER_COLORS["model"], edgecolor="black", linewidth=0.7)
b2 = ax.bar(x+width/2, random_mean, width, yerr=random_std, capsize=4, label="Prior-aware random", color=PAPER_COLORS["random"], edgecolor="black", linewidth=0.7)
annotate_bars(ax, b1, model_mean); annotate_bars(ax, b2, random_mean)
ax.set_xticks(x, [f"Top-{k}" for k in TOPK_LIST]); ax.set_ylabel("Accuracy"); ax.set_title(f"{SUBJECT_ID} Top-k decoding"); ax.grid(axis="y", alpha=0.25); ax.legend(frameon=False); ax.text(0.98, 0.02, "", transform=ax.transAxes, ha="right", va="bottom", fontsize=9, color="0.35")
fig.tight_layout(); save_pdf(fig, FIGURE_DIR, "topk_decoding")

model_mean, model_std = mean_std(fold_metrics["macro_f1"]); random_mean, random_std = mean_std(fold_metrics["random_macro_f1_mean"])
fig, ax = plt.subplots(figsize=(5.8, 4.8)); bars = ax.bar([0, 1], [model_mean, random_mean], yerr=[model_std, random_std], capsize=5, color=[PAPER_COLORS["model"], PAPER_COLORS["random"]], edgecolor="black", linewidth=0.8)
annotate_bars(ax, bars, [model_mean, random_mean]); ax.set_xticks([0, 1], ["Contrastive", "Prior-aware random"]); ax.set_ylabel("Macro-F1"); ax.set_title(f"{SUBJECT_ID} Macro-F1"); ax.grid(axis="y", alpha=0.25)
ax.text(0.5, 0.98, "", transform=ax.transAxes, ha="center", va="top", fontsize=9, color="0.35")
fig.tight_layout(); save_pdf(fig, FIGURE_DIR, "macro_f1")


In [ ]:
def sample_2v2_pairs(labels, n_pairs, seed):
    rng = np.random.default_rng(seed); labels = np.asarray(labels, dtype=np.int64)
    by_class = {int(c): np.where(labels == c)[0] for c in np.unique(labels)}; classes = np.array(list(by_class), dtype=np.int64)
    pos1 = rng.integers(0, len(classes), size=n_pairs); pos2 = (pos1 + rng.integers(1, len(classes), size=n_pairs)) % len(classes)
    class1, class2 = classes[pos1], classes[pos2]; i_idx = np.empty(n_pairs, np.int64); j_idx = np.empty(n_pairs, np.int64)
    for c in classes:
        mask_i, mask_j = class1 == c, class2 == c
        if mask_i.any(): i_idx[mask_i] = rng.choice(by_class[int(c)], size=mask_i.sum(), replace=True)
        if mask_j.any(): j_idx[mask_j] = rng.choice(by_class[int(c)], size=mask_j.sum(), replace=True)
    return i_idx, j_idx

def balanced_2v2(eeg_z, labels, seed):
    eeg_z, prototypes = l2_normalize_np(np.asarray(eeg_z, dtype=np.float32)), l2_normalize_np(class_emb)
    labels = np.asarray(labels, dtype=np.int64); i, j = sample_2v2_pairs(labels, N_2V2_PAIRS, seed); yi, yj = labels[i], labels[j]
    correct = np.sum(eeg_z[i] * prototypes[yi], axis=1) + np.sum(eeg_z[j] * prototypes[yj], axis=1)
    swapped = np.sum(eeg_z[i] * prototypes[yj], axis=1) + np.sum(eeg_z[j] * prototypes[yi], axis=1)
    return float(np.mean((correct > swapped) + 0.5 * (correct == swapped)))

two_vs_two = pd.DataFrame([{"fold_id": item["fold_id"], "accuracy_2v2": balanced_2v2(item["test_z"], item["test_y"], GLOBAL_SEED + 100*item["fold_id"] + 3)} for item in tqdm(fold_results, desc="Balanced 2v2")])
two_mean, two_std = mean_std(two_vs_two["accuracy_2v2"])
two_summary = pd.DataFrame([{"mean": two_mean, "std": two_std, "chance": 0.5, "n_folds": N_FOLDS, "pairs_per_fold": N_2V2_PAIRS}])
two_vs_two.to_csv(RESULTS_DIR / "balanced_2v2_by_fold.csv", index=False); two_summary.to_csv(RESULTS_DIR / "balanced_2v2_summary.csv", index=False)
display(two_vs_two); display(two_summary)
fig, ax = plt.subplots(figsize=(4.8, 4.8)); bars = ax.bar([0], [two_mean], yerr=[two_std], capsize=5, color=PAPER_COLORS["model"], edgecolor="black", linewidth=0.8)
annotate_bars(ax, bars, [two_mean]); ax.axhline(0.5, color="black", linestyle=":", linewidth=1.5, label="Chance = 0.50"); ax.set_xticks([0], ["Balanced 2v2"]); ax.set_ylabel("2v2 accuracy"); ax.set_title(f"{SUBJECT_ID} balanced 2v2")
ax.set_ylim(0.35, min(1.0, two_mean + two_std + 0.12)); ax.grid(axis="y", alpha=0.25); ax.legend(frameon=False); ax.text(0.98, 0.02, "", transform=ax.transAxes, ha="right", va="bottom", fontsize=9, color="0.35")
fig.tight_layout(); save_pdf(fig, FIGURE_DIR, "balanced_2v2")


## MaxCosine reliability analysis

MaxCosine is the raw cosine similarity assigned to the predicted class prototype. Validation-derived thresholds are transferred unchanged to held-out test folds to evaluate coverage transfer and selective Top-1/Top-5 accuracy; validation-referenced quintiles summarize how accuracy changes across the score range, and fold-wise ROC curves quantify discrimination of correct versus incorrect Top-1 predictions. Test labels are used only after thresholds and percentiles are defined from validation scores.


In [ ]:
MAXCOSINE_TARGET_COVERAGES = (1.00, 0.80, 0.60, 0.40, 0.20)
MAXCOSINE_COLORS = {"blue": "#4472C4", "light_blue": "#7EA6D8", "dark_blue": "#2F5597", "green": "#4DAF7C", "orange": "#E19A25", "gray": "#666666"}
MAXCOSINE_FIGURE_DIR = FIGURE_DIR / "maxcosine"; MAXCOSINE_FIGURE_DIR.mkdir(exist_ok=True)

def maxcosine(cosine, ranked): return np.asarray(cosine)[np.arange(len(cosine)), np.asarray(ranked)[:, 0]].astype(np.float32)
def threshold_for_coverage(scores, coverage):
    scores = np.asarray(scores, dtype=float); n_keep = int(np.clip(np.ceil(coverage * len(scores)), 1, len(scores)))
    return float(np.sort(scores)[::-1][n_keep - 1])
def validation_percentile(test_scores, validation_scores):
    reference = np.sort(np.asarray(validation_scores, dtype=float))
    return (np.searchsorted(reference, np.asarray(test_scores), side="right") / len(reference)).astype(np.float32)
def topk_hits(y, ranked, k): return (np.asarray(ranked)[:, :k] == np.asarray(y)[:, None]).any(axis=1)

trial_rows, selective_rows, auroc_rows, maxcosine_cache = [], [], [], {}
for item in tqdm(fold_results, desc="MaxCosine analysis"):
    fold_id = item["fold_id"]; val_score = maxcosine(item["val_cosine"], item["val_ranked"]); test_score = maxcosine(item["test_cosine"], item["test_ranked"])
    y = item["test_y"]; pred = item["test_ranked"][:, 0]; correct = pred == y; top5 = topk_hits(y, item["test_ranked"], 5); percentile = validation_percentile(test_score, val_score)
    thresholds = {coverage: threshold_for_coverage(val_score, coverage) for coverage in MAXCOSINE_TARGET_COVERAGES}
    auroc_rows.append({"fold_id": fold_id, "auroc": roc_auc_score(correct.astype(int), test_score) if np.unique(correct).size == 2 else np.nan})
    for local_i, trial_uid in enumerate(item["test_idx"]):
        trial_rows.append({"fold_id": fold_id, "local_test_index": local_i, "trial_uid": int(trial_uid), "true_class_id": int(y[local_i]), "predicted_class_id": int(pred[local_i]),
                           "top1_correct": bool(correct[local_i]), "top5_correct": bool(top5[local_i]), "maxcosine": float(test_score[local_i]), "validation_percentile": float(percentile[local_i])})
    for target, threshold in thresholds.items():
        val_keep, test_keep = val_score >= threshold, test_score >= threshold
        selective_rows.append({"fold_id": fold_id, "target_validation_coverage": target, "validation_threshold": threshold, "validation_coverage": val_keep.mean(), "test_coverage": test_keep.mean(),
                               "top1_accuracy": correct[test_keep].mean() if test_keep.any() else np.nan, "top5_accuracy": top5[test_keep].mean() if test_keep.any() else np.nan})
    maxcosine_cache[fold_id] = {"validation_score": val_score, "test_score": test_score, "test_percentile": percentile}

maxcosine_trials = pd.DataFrame(trial_rows); selective_by_fold = pd.DataFrame(selective_rows); auroc_by_fold = pd.DataFrame(auroc_rows)
summary_rows = []
for target, group in selective_by_fold.groupby("target_validation_coverage", sort=False):
    row = {"target_validation_coverage": target}
    for column in ["validation_threshold", "validation_coverage", "test_coverage", "top1_accuracy", "top5_accuracy"]:
        row[f"{column}_mean"], row[f"{column}_std"] = mean_std(group[column])
    summary_rows.append(row)
selective_summary = pd.DataFrame(summary_rows).sort_values("target_validation_coverage", ascending=False)
maxcosine_trials["quintile"] = pd.cut(maxcosine_trials["validation_percentile"], bins=np.linspace(0, 1, 6), labels=["Q1", "Q2", "Q3", "Q4", "Q5"], include_lowest=True)
quintile_by_fold = maxcosine_trials.groupby(["fold_id", "quintile"], observed=True).agg(n=("top1_correct", "size"), top1_accuracy=("top1_correct", "mean")).reset_index()
quintile_rows = []
for quintile, group in quintile_by_fold.groupby("quintile", observed=True, sort=False):
    mean, std = mean_std(group["top1_accuracy"]); quintile_rows.append({"quintile": str(quintile), "n_test": int(group["n"].sum()), "top1_accuracy_mean": mean, "top1_accuracy_std": std})
quintile_summary = pd.DataFrame(quintile_rows)
selective_summary.to_csv(RESULTS_DIR / "maxcosine_selective_summary.csv", index=False); quintile_summary.to_csv(RESULTS_DIR / "maxcosine_quintile_summary.csv", index=False); auroc_by_fold.to_csv(RESULTS_DIR / "maxcosine_auroc_by_fold.csv", index=False)
display(selective_summary); display(quintile_summary); display(auroc_by_fold)


In [ ]:
# Larger typography for figures that will be scaled to a two-column paper
TITLE_SIZE = 15
LABEL_SIZE = 13
TICK_SIZE = 11.5
LEGEND_SIZE = 10.5
ANNOTATION_SIZE = 11

def style_ax(ax, title, xlabel, ylabel):
    ax.set_title(title, fontsize=TITLE_SIZE, pad=10)
    ax.set_xlabel(xlabel, fontsize=LABEL_SIZE, labelpad=7)
    ax.set_ylabel(ylabel, fontsize=LABEL_SIZE, labelpad=7)
    ax.tick_params(axis="both", labelsize=TICK_SIZE)

operating = selective_summary.sort_values("target_validation_coverage")
x = operating["target_validation_coverage"].to_numpy(float)

fig, ax = plt.subplots(figsize=(7.2, 5.2))
ax.plot([0, 1], [0, 1], ":", color=MAXCOSINE_COLORS["gray"], linewidth=2.2, label="Ideal")
ax.errorbar(
    x, operating["validation_coverage_mean"],
    yerr=operating["validation_coverage_std"].fillna(0),
    marker="o", markersize=7, linewidth=2.2, capsize=4,
    color=MAXCOSINE_COLORS["green"], label="Validation coverage"
)
ax.errorbar(
    x, operating["test_coverage_mean"],
    yerr=operating["test_coverage_std"].fillna(0),
    marker="s", markersize=7, linewidth=2.2, capsize=4,
    color=MAXCOSINE_COLORS["blue"], label="Held-out test coverage"
)
style_ax(ax, "Validation-to-test coverage transfer", "Target validation coverage", "Achieved coverage")
ax.set_xlim(0.15, 1.02)
ax.set_ylim(0.15, 1.02)
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.grid(alpha=0.22)
ax.legend(frameon=False, fontsize=LEGEND_SIZE)
fig.tight_layout()
save_pdf(fig, MAXCOSINE_FIGURE_DIR, "coverage_transfer")


plot_data = selective_summary.sort_values("test_coverage_mean")
x = plot_data["test_coverage_mean"].to_numpy(float)

fig, ax = plt.subplots(figsize=(7.2, 5.2))
ax.errorbar(
    x, plot_data["top1_accuracy_mean"],
    xerr=plot_data["test_coverage_std"].fillna(0),
    yerr=plot_data["top1_accuracy_std"].fillna(0),
    marker="o", markersize=7, linewidth=2.3, capsize=4,
    color=MAXCOSINE_COLORS["blue"], label="Top-1"
)
ax.errorbar(
    x, plot_data["top5_accuracy_mean"],
    xerr=plot_data["test_coverage_std"].fillna(0),
    yerr=plot_data["top5_accuracy_std"].fillna(0),
    marker="s", markersize=7, linestyle="--", linewidth=2.3, capsize=4,
    color=MAXCOSINE_COLORS["orange"], label="Top-5"
)
annotate_points(ax, x, plot_data["top1_accuracy_mean"], dy=-16)
annotate_points(ax, x, plot_data["top5_accuracy_mean"], dy=10)
style_ax(ax, "Top-1 and Top-5 accuracy–coverage", "Held-out test coverage", "Accepted accuracy")
ax.set_xlim(0.15, 1.02)
ax.set_ylim(0, 1)
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.grid(alpha=0.22)
ax.legend(frameon=False, fontsize=LEGEND_SIZE)
for text in ax.texts:
    text.set_fontsize(ANNOTATION_SIZE)
fig.tight_layout()
save_pdf(fig, MAXCOSINE_FIGURE_DIR, "selective_accuracy_coverage")


quintile_plot = quintile_summary.copy()
x = np.arange(len(quintile_plot))

fig, ax = plt.subplots(figsize=(7.0, 5.0))
ax.errorbar(
    x, quintile_plot["top1_accuracy_mean"],
    yerr=quintile_plot["top1_accuracy_std"].fillna(0),
    marker="o", markersize=9, linewidth=2.3, capsize=4,
    color=MAXCOSINE_COLORS["blue"]
)
annotate_points(ax, x, quintile_plot["top1_accuracy_mean"], dy=10)
ax.set_xticks(x, ["Q1\nLowest", "Q2", "Q3", "Q4", "Q5\nHighest"])
style_ax(
    ax,
    "Top-1 accuracy across MaxCosine quintiles",
    "Validation-referenced MaxCosine quintile",
    "Top-1 accuracy"
)
ax.set_ylim(0, 0.25)
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.grid(axis="y", alpha=0.22)
for text in ax.texts:
    text.set_fontsize(ANNOTATION_SIZE)
fig.tight_layout()
save_pdf(fig, MAXCOSINE_FIGURE_DIR, "maxcosine_quintiles")


fig, ax = plt.subplots(figsize=(7.4, 6.6))
fpr_grid = np.linspace(0, 1, 501)
fold_tprs, fold_aucs = [], []

ax.plot(
    [0, 1], [0, 1], "--",
    color=MAXCOSINE_COLORS["gray"],
    linewidth=2.2,
    label="Chance (AUROC = 0.500)"
)

for fold_id in sorted(maxcosine_trials["fold_id"].unique()):
    fold = maxcosine_trials[maxcosine_trials["fold_id"] == fold_id]
    truth = fold["top1_correct"].astype(int).to_numpy()
    scores = fold["maxcosine"].to_numpy(float)

    fpr, tpr, _ = roc_curve(truth, scores)
    auc = float(roc_auc_score(truth, scores))

    fold_aucs.append(auc)
    fold_tprs.append(np.interp(fpr_grid, fpr, tpr))
    ax.plot(fpr, tpr, linewidth=1.8, alpha=0.75, label=f"Fold {fold_id} ({auc:.3f})")

fold_tprs = np.asarray(fold_tprs)
mean_tpr = fold_tprs.mean(axis=0)
std_tpr = fold_tprs.std(axis=0, ddof=1)
mean_tpr[0], mean_tpr[-1] = 0, 1
mean_auc, std_auc = mean_std(fold_aucs)

ax.fill_between(
    fpr_grid,
    np.clip(mean_tpr - std_tpr, 0, 1),
    np.clip(mean_tpr + std_tpr, 0, 1),
    color=MAXCOSINE_COLORS["light_blue"],
    alpha=0.20,
    label="Fold mean ± std"
)
ax.plot(
    fpr_grid, mean_tpr,
    color=MAXCOSINE_COLORS["dark_blue"],
    linewidth=3.2,
    label=f"Mean AUROC = {mean_auc:.3f} ± {std_auc:.3f}"
)
style_ax(
    ax,
    "MaxCosine discrimination of prediction correctness",
    "False-positive rate",
    "True-positive rate"
)
ax.set_xlim(-0.01, 1.01)
ax.set_ylim(-0.01, 1.01)
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.grid(alpha=0.22)
ax.legend(frameon=False, fontsize=LEGEND_SIZE, loc="lower right")
fig.tight_layout()
save_pdf(fig, MAXCOSINE_FIGURE_DIR, "correctness_roc")

## High-confidence predictions

For qualitative inspection, five held-out trials are selected near each validation-referenced percentile target: 100%, 90%, and 80%. Selection is performed independently within each fold, yielding 15 examples in total.


In [ ]:
EXAMPLE_PERCENTILES = {"P100": 1.00, "P90": 0.90, "P80": 0.80}
EXAMPLE_TOP_N = 5
EXAMPLE_TRANSLATION_CACHE_PATH = CACHE_DIR / "example_trial_translations.json"

def load_example_translation_cache():
    try:
        with open(EXAMPLE_TRANSLATION_CACHE_PATH, "r", encoding="utf-8") as f: return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError): return {}

EXAMPLE_TRANSLATION_CACHE = load_example_translation_cache()

TRANSLATION_CACHE_PATH = CACHE_DIR / "translation_cache.json"

try:
    with open(TRANSLATION_CACHE_PATH, "r", encoding="utf-8") as f:
        TRANSLATION_CACHE = json.load(f)
except (FileNotFoundError, json.JSONDecodeError):
    TRANSLATION_CACHE = {}

def translate_sentence(text_zh):
    text_zh = str(text_zh).strip()
    if text_zh in TEXT_TRANSLATIONS:
        return TEXT_TRANSLATIONS[text_zh]
    if text_zh in TRANSLATION_CACHE:
        return TRANSLATION_CACHE[text_zh]

    from deep_translator import GoogleTranslator
    translated = GoogleTranslator(source="zh-CN", target="en").translate(text_zh)
    if not translated:
        raise RuntimeError(f"Could not translate: {text_zh}")

    TRANSLATION_CACHE[text_zh] = translated
    with open(TRANSLATION_CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(TRANSLATION_CACHE, f, ensure_ascii=False, indent=2)
    return translated

def build_example_pool():
    rows = []
    for item in fold_results:
        fold_id = item["fold_id"]
        cache = maxcosine_cache[fold_id]
        for local_i, trial_uid in enumerate(item["test_idx"]):
            top_ids = item["test_ranked"][local_i, :EXAMPLE_TOP_N].astype(int)
            rows.append({
                "fold_id": fold_id,
                "trial_uid": int(trial_uid),
                "sentence_zh": trial_index.iloc[int(trial_uid)]["sentence_zh"],
                "true_class_id": int(item["test_y"][local_i]),
                "predicted_class_id": int(top_ids[0]),
                "maxcosine": float(cache["test_score"][local_i]),
                "validation_percentile": float(cache["test_percentile"][local_i]),
                "top_ids": top_ids,
                "top_cosine": item["test_cosine"][local_i, top_ids].astype(float)
            })
    return pd.DataFrame(rows)

def choose_examples(pool):
    selected = []
    for label, target in EXAMPLE_PERCENTILES.items():
        for fold_id, fold in pool.groupby("fold_id", sort=True):
            chosen = fold.assign(distance=(fold["validation_percentile"]-target).abs()).sort_values(["distance", "trial_uid"], kind="mergesort").iloc[0].copy()
            scores = maxcosine_cache[int(fold_id)]["validation_score"]
            try: reference = float(np.quantile(scores, target, method="nearest"))
            except TypeError: reference = float(np.quantile(scores, target, interpolation="nearest"))
            chosen["percentile_label"] = label
            chosen["target_percentile"] = target
            chosen["reference_score"] = reference
            selected.append(chosen)
    result = pd.DataFrame(selected)
    result["example_index"] = result.groupby("percentile_label").cumcount() + 1
    result["sentence_en"] = result["sentence_zh"].map(translate_sentence)
    return result

examples = choose_examples(build_example_pool())

summary_rows, top5_rows = [], []
for row in examples.itertuples():
    summary_rows.append({
        "percentile_label": row.percentile_label,
        "example_index": row.example_index,
        "fold_id": row.fold_id,
        "trial_uid": row.trial_uid,
        "sentence_zh": row.sentence_zh,
        "sentence_en": row.sentence_en,
        "true_class_id": row.true_class_id,
        "true_class_en": CLASS_ID_TO_EN[row.true_class_id],
        "predicted_class_id": row.predicted_class_id,
        "predicted_class_en": CLASS_ID_TO_EN[row.predicted_class_id],
        "top1_correct": row.true_class_id == row.predicted_class_id,
        "maxcosine": row.maxcosine,
        "validation_percentile": row.validation_percentile,
        "target_percentile": row.target_percentile,
        "reference_score": row.reference_score
    })
    for rank, (class_id, cosine) in enumerate(zip(row.top_ids, row.top_cosine), 1):
        top5_rows.append({
            "percentile_label": row.percentile_label,
            "example_index": row.example_index,
            "fold_id": row.fold_id,
            "trial_uid": row.trial_uid,
            "rank": rank,
            "predicted_class_id": int(class_id),
            "predicted_class_en": CLASS_ID_TO_EN[int(class_id)],
            "eeg_to_class_cosine": float(cosine),
            "is_true_class": int(class_id) == row.true_class_id
        })

example_summary = pd.DataFrame(summary_rows)
example_top5 = pd.DataFrame(top5_rows)
example_summary.to_csv(RESULTS_DIR / "maxcosine_examples.csv", index=False)
example_top5.to_csv(RESULTS_DIR / "maxcosine_examples_top5.csv", index=False)

In [ ]:
PERCENTILE_TITLES = {"P100": "100th percentile", "P90": "90th percentile", "P80": "80th percentile"}

def print_examples(examples):
    for label in EXAMPLE_PERCENTILES:
        group = examples[examples["percentile_label"] == label].sort_values("example_index")
        print(f"\n{'='*88}\n{PERCENTILE_TITLES[label].upper()}\n{'='*88}")
        for row in group.itertuples():
            correct = int(row.predicted_class_id) == int(row.true_class_id)
            print(f"\nExample {int(row.example_index)}/5 | Fold {int(row.fold_id)} | Trial {int(row.trial_uid)}")
            print(f"Sentence: {row.sentence_en}")
            print(f"True category: {int(row.true_class_id):02d} · {CLASS_ID_TO_EN[int(row.true_class_id)]}")
            print(f"Top-1: {int(row.predicted_class_id):02d} · {CLASS_ID_TO_EN[int(row.predicted_class_id)]} | {'correct' if correct else 'incorrect'}")
            print(f"MaxCosine: {row.maxcosine:.3f} | Validation percentile: {row.validation_percentile:.1%} | Reference: {row.reference_score:.3f}")
            print("Top-5 predictions:")
            for rank, (class_id, cosine) in enumerate(zip(row.top_ids, row.top_cosine), 1):
                marker = "  <- true class" if int(class_id) == int(row.true_class_id) else ""
                print(f"  {rank}. {int(class_id):02d} · {CLASS_ID_TO_EN[int(class_id)]} | cosine = {float(cosine):.3f}{marker}")

print_examples(examples)

## Semantic Gallery Probe

The Semantic Gallery Probe compares held-out EEG embeddings with a frozen gallery of unseen candidate sentences from five semantic domains: Food, Sports, Nature, Transport, and Emotions. For each domain, the two highest-confidence held-out trials are selected across all test folds using the model's MaxCosine score. Each selected EEG embedding is then compared with every gallery sentence using cosine similarity, providing a qualitative view of its semantic alignment with unseen text candidates.

In [ ]:
SGP_TARGET_IDS = [5, 38, 2, 7, 27]
SGP_TOP_EXAMPLES = 2
SGP_FIGURE_DIR = FIGURE_DIR / "semantic_gallery_probe"; SGP_FIGURE_DIR.mkdir(exist_ok=True)
SGP_TRANSLATION_CACHE_PATH = CACHE_DIR / "sgp_trial_translations.json"
SGP_GALLERY = {
    5: {"name": "Food", "sentences": [
        ("我非常饿，想吃大餐", "I am very hungry"), ("这家餐厅的面条很好吃", "Tasty noodles"),
        ("正在厨房里做饭", "Cooking in kitchen"), ("美味的晚餐", "Delicious dinner"),
        ("喝一杯热咖啡", "Hot coffee"), ("甜点是巧克力蛋糕", "Chocolate cake")]},
    38: {"name": "Sports", "sentences": [
        ("篮球比赛非常激烈", "Intense basketball"), ("奥运会金牌", "Olympic gold medal"),
        ("去健身房举重", "Lifting weights"), ("足球运动员射门", "Soccer player shoots"),
        ("在游泳池游泳", "Swimming in pool"), ("每天早上跑步", "Running every morning")]},
    2: {"name": "Nature", "sentences": [
        ("窗外正在下大雨", "Heavy rain outside"), ("森林里的空气很清新", "Fresh forest air"),
        ("美丽的花朵盛开了", "Flowers blooming"), ("冬天下了很多雪", "Snow in winter"),
        ("太阳从东方升起", "Sun rises in east"), ("秋天的树叶变黄了", "Autumn leaves yellow")]},
    7: {"name": "Transport", "sentences": [
        ("我开车去上班", "Driving to work"), ("火车准时到达", "Train arrived on time"),
        ("飞机场很大", "Airport is huge"), ("乘坐公共汽车", "Riding the bus"),
        ("交通非常拥堵", "Traffic is heavy"), ("出租车司机", "Taxi driver")]},
    27: {"name": "Emotions", "sentences": [
        ("我感到非常快乐", "I feel very happy"), ("因为悲伤而哭泣", "Crying (Sadness)"),
        ("对这个结果很生气", "Angry about result"), ("充满爱意", "Full of love"),
        ("心情很沮丧", "Feeling depressed"), ("惊喜的礼物", "Surprise gift")]}
}
try:
    with open(SGP_TRANSLATION_CACHE_PATH, "r", encoding="utf-8") as f:
        SGP_TRANSLATION_CACHE = json.load(f)
except (FileNotFoundError, json.JSONDecodeError):
    SGP_TRANSLATION_CACHE = {}

def build_sgp_gallery():
    rows = [(cid, zh, en) for cid in sorted(SGP_GALLERY) for zh, en in SGP_GALLERY[cid]["sentences"]]
    ids = np.array([r[0] for r in rows], dtype=np.int64)
    zh, en = [r[1] for r in rows], [r[2] for r in rows]
    path = CACHE_DIR / "sgp_gallery_embeddings.npz"; vectors = None
    if path.exists():
        data = np.load(path, allow_pickle=True)
        if data["sentences_zh"].tolist() == zh: vectors = data["vectors"].astype(np.float32)
    if vectors is None:
        model = SentenceTransformer(TEXT_MODEL_NAME, device=TEXT_EMBEDDING_DEVICE)
        vectors = l2_normalize_np(
            model.encode(zh, convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
        )
        np.savez(path, sentences_zh=np.array(zh, dtype=object), vectors=vectors)
    return en, ids, vectors, PCA(n_components=2).fit_transform(vectors)

def select_high_confidence_sgp_trials(target_id, n=2):
    candidates = []
    for item in fold_results:
        fold = int(item["fold_id"]); cache = maxcosine_cache[fold]
        for i in np.flatnonzero(item["test_y"] == target_id):
            pred = int(item["test_ranked"][i, 0])
            candidates.append({
                "fold_id": fold, "trial_uid": int(item["test_idx"][i]),
                "predicted_class_id": pred, "top1_correct": pred == target_id,
                "maxcosine": float(cache["test_score"][i]),
                "eeg_z": l2_normalize_np(item["test_z"][i:i+1])[0]
            })
    return sorted(candidates, key=lambda r: (-r["maxcosine"], r["fold_id"], r["trial_uid"]))[:n]

def plot_sgp(target_id, rank, example, gallery_en, gallery_ids, gallery_vectors, xy):
    activations = gallery_vectors @ example["eeg_z"]
    offsets = {5: (0.04, -0.03), 38: (-0.03, 0.07), 2: (0, 0.09), 7: (0, -0.02), 27: (0, 0.11)}
    fig, ax = plt.subplots(figsize=(11, 9))
    for cid in sorted(set(gallery_ids)):
        idx = np.where(gallery_ids == cid)[0]; points = xy[idx]
        color = "#e74c3c" if cid == target_id else "#95a5a6"
        if len(points) >= 3:
            hull = ConvexHull(points)
            ax.add_patch(plt.Polygon(points[hull.vertices], facecolor=color, alpha=0.1, zorder=0))
            center = points.mean(0); dx, dy = offsets[cid]
            ax.text(center[0]+dx, center[1]+dy, SGP_GALLERY[cid]["name"].upper(),
                    fontsize=14, color=color, alpha=0.3, ha="center", va="center")

    scatter = ax.scatter(
        xy[:, 0], xy[:, 1], c=activations, cmap="RdYlBu_r",
        s=250, edgecolors="white", linewidth=1.5, alpha=0.9, zorder=10
    )
    labels = [
        ax.text(xy[i, 0], xy[i, 1], label,
                fontsize=12 if gallery_ids[i] == target_id else 10,
                fontweight="bold" if gallery_ids[i] == target_id else "normal",
                alpha=1 if gallery_ids[i] == target_id else 0.5,
                ha="center", va="center", zorder=15)
        for i, label in enumerate(gallery_en)
    ]
    adjust_text(labels, ax=ax, arrowprops=dict(arrowstyle="-", color="gray", alpha=0.3, lw=0.5))

    cbar = fig.colorbar(scatter, ax=ax)
    cbar.set_label("Cosine similarity", rotation=270, labelpad=18, fontsize=15)
    cbar.ax.tick_params(labelsize=12)

    uid = example["trial_uid"]
    sentence_en = translate_sentence(trial_index.iloc[uid]["sentence_zh"])

    fig.text(0.5, 0.98, "Semantic Gallery Probe", ha="center", va="top", fontsize=22)
    fig.text(0.5, 0.93, f"Target: {SGP_GALLERY[target_id]['name']}", ha="center", va="top", fontsize=17)
    fig.text(0.5, 0.895, f'Subject imagining: "{sentence_en}"',
             ha="center", va="top", fontsize=15, color="#34495e")

    ax.set_xlabel("PC1", fontsize=16); ax.set_ylabel("PC2", fontsize=16)
    ax.tick_params(axis="both", labelsize=12); ax.grid(alpha=0.18)
    fig.tight_layout(rect=[0, 0.03, 1, 0.86])
    
    filename = f"sgp_{SGP_GALLERY[target_id]['name'].lower()}_example_{rank}"
    return save_pdf(fig, SGP_FIGURE_DIR, filename), sentence_en

In [ ]:
gallery_en, gallery_ids, gallery_vectors, gallery_xy = build_sgp_gallery()
sgp_rows = []
for target_id in SGP_TARGET_IDS:
    name = SGP_GALLERY[target_id]["name"]
    selected = select_high_confidence_sgp_trials(target_id, SGP_TOP_EXAMPLES)
    if not selected:
        print(f"No held-out {name} trials found.")
        continue
    print(f"\n{name}: selected {len(selected)} highest-confidence held-out trials")
    for rank, ex in enumerate(selected, 1):
        path, sentence_en = plot_sgp(
            target_id, rank, ex,
            gallery_en, gallery_ids, gallery_vectors, gallery_xy)
        uid = ex["trial_uid"]
        pred = ex["predicted_class_id"]
        sgp_rows.append({
            "target_id": target_id, "target_name": name, "confidence_rank": rank,
            "fold_id": ex["fold_id"], "trial_uid": uid,
            "sentence_zh": trial_index.iloc[uid]["sentence_zh"], "sentence_en": sentence_en,
            "maxcosine": ex["maxcosine"], "predicted_class_id": pred,
            "predicted_class_name": CLASS_ID_TO_EN[pred],
            "top1_correct": ex["top1_correct"], "figure": path.name
        })
        print(f"  Example {rank}: MaxCosine={ex['maxcosine']:.3f} | "
            f"Fold {ex['fold_id']} | Trial {uid} | "
            f"Top-1 {'correct' if ex['top1_correct'] else 'incorrect'}")
sgp_summary = pd.DataFrame(sgp_rows)
sgp_summary.to_csv(RESULTS_DIR / "semantic_gallery_probe_summary.csv", index=False)
display(sgp_summary)
print(f"\nSGP complete: generated {len(sgp_summary)} figures.")

## Generalization across semantic axes

As an exploratory post hoc analysis, we assess whether the learned EEG representation preserves broad semantic structure beyond the original 39-class taxonomy. Five semantic axes are introduced only at evaluation time, each defined by two sets of Chinese text anchors. For every held-out trial, the EEG embedding and its corresponding imagined sentence are independently assigned to one side of the same axis. Agreement is summarized using balanced accuracy.

In [ ]:
from sklearn.metrics import balanced_accuracy_score

SEMANTIC_AXES = {
    "Practical / Transactional ↔ Social / Emotional": {
        "zh": (["实际事务和安排", "付款、价格或服务相关的实际问题", "需要解决的日常事务"],
               ["人与人之间的情感交流", "社交关系和情绪表达", "涉及感受或人际互动的事情"]),
        "en": (["practical affairs and arrangements", "payments, prices, or service-related matters", "everyday practical matters to resolve"],
               ["emotional communication between people", "social relationships and emotional expression", "feelings or interpersonal interaction"])},
    "Physical / Embodied ↔ Informational / Cognitive": {
        "zh": (["与身体或身体状态有关的事情", "涉及饮食、健康或身体活动", "身体动作和感官体验"],
               ["与知识和信息有关的事情", "涉及学习、理解或获取信息", "认知、教育或信息处理"]),
        "en": (["something related to the body or bodily state", "food, health, or physical activity", "bodily actions and sensory experience"],
               ["something related to knowledge and information", "learning, understanding, or obtaining information", "cognition, education, or information processing"])},
    "Goal-directed / Obligation ↔ Leisure / Enjoyment": {
        "zh": (["为了完成任务或目标而做的事情", "与工作、计划或责任有关的活动", "有明确目的或义务的行为"],
               ["为了娱乐和享受而进行的活动", "休闲、兴趣或度假活动", "没有工作义务的愉快活动"]),
        "en": (["something done to complete a task or goal", "work, planning, or responsibility", "goal-directed or obligatory behavior"],
               ["activities for entertainment and enjoyment", "leisure, hobbies, or vacation", "pleasant activity without work obligations"])},
    "Mobility / Outside-world ↔ Home / Personal-life": {
        "zh": (["在外部环境中移动或旅行", "交通、出行或户外环境", "离开个人空间并在外界活动"],
               ["个人生活和日常私人事务", "家庭、住房或个人生活环境", "发生在个人或家庭生活中的事情"]),
        "en": (["moving or travelling through the outside world", "transportation, travel, or outdoor environments", "activity outside one's personal space"],
               ["personal life and everyday private matters", "home, housing, or personal living environment", "things happening in personal or family life"])},
    "Individual / Self-related ↔ Interpersonal / Social": {
        "zh": (["主要与自己有关的个人体验", "个人状态、需求或行为", "不需要与其他人互动的事情"],
               ["涉及其他人的互动", "人与人之间的交流或关系", "社会互动和共同活动"]),
        "en": (["a personal experience mainly involving oneself", "one's own state, needs, or behavior", "something not requiring interaction with others"],
               ["interaction involving other people", "communication or relationships between people", "social interaction and shared activities"])}
}

text_model = SentenceTransformer(TEXT_MODEL_NAME, device=TEXT_EMBEDDING_DEVICE)
def anchor(phrases):
    z = text_model.encode(phrases, convert_to_numpy=True, normalize_embeddings=True)
    return l2_normalize_np(z.mean(0, keepdims=True))[0]
axis_vectors = {k: tuple(anchor(x) for x in v["zh"]) for k, v in SEMANTIC_AXES.items()}
rows = []
for item in fold_results:
    sentences = [trial_index.iloc[int(i)]["sentence_zh"] for i in item["test_idx"]]
    eeg = l2_normalize_np(item["test_z"])
    text = l2_normalize_np(np.vstack([SENTENCE_TO_EMBEDDING[s] for s in sentences]))
    for axis, (left, right) in axis_vectors.items():
        ts, es = (text @ left - text @ right) > 0, (eeg @ left - eeg @ right) > 0
        rows.extend({"axis": axis, "fold_id": int(item["fold_id"]), "sentence_zh": s,
                     "text_side": bool(t), "eeg_side": bool(e), "correct": bool(t == e)}
                    for s, t, e in zip(sentences, ts, es))
attribute_trials = pd.DataFrame(rows)
N_BOOT = 10_000
rng = np.random.default_rng(GLOBAL_SEED)
def bootstrap_ci(d):
    g = (d.groupby(["fold_id", "sentence_zh"], sort=False)
          .agg(text_side=("text_side", "first"), n=("correct", "size"), correct=("correct", "sum"))
          .reset_index())
    totals = {False: np.zeros((N_BOOT, 2)), True: np.zeros((N_BOOT, 2))}

    for (_, side), x in g.groupby(["fold_id", "text_side"], sort=False):
        a = x[["n", "correct"]].to_numpy()
        totals[bool(side)] += a[rng.integers(len(a), size=(N_BOOT, len(a)))].sum(1)

    boot = 0.5 * sum(v[:, 1] / v[:, 0] for v in totals.values())
    return np.quantile(boot, [0.025, 0.975])
results = []
for axis in SEMANTIC_AXES:
    d = attribute_trials[attribute_trials["axis"] == axis]
    lo, hi = bootstrap_ci(d)
    results.append({
        "axis": axis, "n_trials": len(d), "n_sentences": d["sentence_zh"].nunique(),
        "raw_agreement": d["correct"].mean(),
        "balanced_accuracy": balanced_accuracy_score(d["text_side"], d["eeg_side"]),
        "ci_low": lo, "ci_high": hi
    })
attribute_results = pd.DataFrame(results)
attribute_results.to_csv(RESULTS_DIR / "semantic_axis_generalization.csv", index=False)
display(attribute_results.round(4))

In [ ]:
SHORT_LABELS = {
    "Practical / Transactional ↔ Social / Emotional": "Practical ↔ Emotional",
    "Physical / Embodied ↔ Informational / Cognitive": "Physical ↔ Cognitive",
    "Goal-directed / Obligation ↔ Leisure / Enjoyment": "Goal-directed ↔ Leisure",
    "Mobility / Outside-world ↔ Home / Personal-life": "Mobility ↔ Personal life",
    "Individual / Self-related ↔ Interpersonal / Social": "Individual ↔ Social"
}

x = np.arange(len(attribute_results))
acc = attribute_results["balanced_accuracy"].to_numpy()
lo, hi = attribute_results["ci_low"].to_numpy(), attribute_results["ci_high"].to_numpy()

fig, ax = plt.subplots(figsize=(8.4, 5.4))
bars = ax.bar(x, acc, width=0.62, yerr=np.vstack([acc - lo, hi - acc]), capsize=4)
ax.axhline(0.5, color="0.4", ls="--", lw=1.6, label="Chance (50%)")

ax.set_xticks(x, [SHORT_LABELS[a] for a in attribute_results["axis"]], rotation=22, ha="right")
ax.set_ylim(0, max(0.72, hi.max() + 0.05))
ax.set_ylabel("Balanced accuracy", fontsize=13)
ax.set_title("Generalization across semantic axes", fontsize=15)
ax.tick_params(axis="both", labelsize=11)
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.grid(axis="y", alpha=0.2)
ax.legend(frameon=False, fontsize=10.5)

for bar, value in zip(bars, acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.012,
            f"{value:.1%}", ha="center", va="bottom", fontsize=10)
fig.tight_layout()
save_pdf(fig, FIGURE_DIR, "semantic_axis_generalization")

In [ ]:
print(f"Completed {SUBJECT_ID}.\nResults: {RESULTS_DIR}\nFigures: {FIGURE_DIR}")